# EPIC-Rec walkthrough

Trains the path-additive model on the synthetic dataset, explains the removal of one recommendation under the three settings, and inspects the verified explanation objects. Replace `make_synthetic` by `load_or_prepare(load_config('configs/movielens.yaml'))` for real data (after `epic-rec download` / `prepare`).

In [ ]:
from epic_rec.data.synthetic import make_synthetic
from epic_rec.models import PathAdditiveRecommender
from epic_rec.optimization import ExplanationInstance
from epic_rec.explainers import explain
from epic_rec.paths import top_paths

ds = make_synthetic(seed=0)
model = PathAdditiveRecommender.fit(ds, {'seed': 0})
ds.stats(), model.theta, model.beta

## A user's list and the evidence for the target

In [ ]:
u = 7
inst = ExplanationInstance(model, u, K=10, alpha=0.5)
print('top-10:', inst.L.tolist())
t = int(inst.L[1])
inst.set_target(t)
for p in top_paths(model, u, t, None, 5):
    print(p)

## Minimum masks under C1, C1 + C2 and C1 + C2 + C3

In [ ]:
results = {s: explain(inst, s, 'exact') for s in ('A', 'B', 'C')}
for s, e in results.items():
    print(e.summary())

## The verification report

`success` is true only if the independent verifier (scores recomputed from the model, path decomposition checked, evidence paths re-enumerated) confirms every requested constraint.

In [ ]:
e = results['C']
v = e.verification
{k: v[k] for k in ('passed', 'passed_P', 'passed_P_eps', 'c1', 'c2', 'c3', 'top_k_after', 'entered', 'overlap', 'kendall_retained', 'evidence_survival')}

In [ ]:
for rec in (v['evidence'] or [])[:3]:
    print(rec['item'], 'faithful:', rec['faithful'], 'strongest:', rec['strongest'])

## Approximate methods on the same request

In [ ]:
import numpy as np
for m in ('greedy:violation', 'greedy:target_score', 'beam', 'continuous'):
    print(explain(inst, 'C', m, rng=np.random.default_rng(0)).summary())